Derived from [nuImages Tutorial](https://github.com/nutonomy/nuscenes-devkit/blob/master/python-sdk/tutorials/nuimages_tutorial.ipynb)

In [201]:
NUIMAGES_ROOT="/home/tensorturtle/DatasetsPublic/nuimages-full"

The above directory should look like:

```
NUIMAGES_ROOT
├── nuimages-v1.0-all-metadata.tgz
└── nuimages-v1.0-all-samples.tgz
```

Untar them:

```
tar -xvf nuimages-v1.0-all-metadata.tgz
tar -xvf nuimages-v1.0-all-samples.tgz
```


In [202]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from pathlib import Path
from nuimages import NuImages
from tqdm import tqdm
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Categories 

In [17]:
nuim = NuImages(dataroot=Path(NUIMAGES_ROOT).resolve(), version=f"v1.0-train", verbose=False, lazy=True)
for d in nuim.category:
    print(f"{d['name']} : {d['description']}")

animal : All animals, e.g. cats, rats, dogs, deer, birds.
flat.driveable_surface : Surfaces should be regarded with no concern of traffic rules, lanes etc. Exclude any road obstacles. This includes paved and unpaved surfaces
human.pedestrian.adult : Adult subcategory.
human.pedestrian.child : Child subcategory.
human.pedestrian.construction_worker : Construction worker
human.pedestrian.personal_mobility : A small electric or self-propelled vehicle, e.g. skateboard, segway, or scooters, on which the person typically travels in a upright position. Driver and (if applicable) rider should be included in the bounding box along with the vehicle.
human.pedestrian.police_officer : Police officer.
human.pedestrian.stroller : Strollers. If a person is in the stroller, include in the annotation.
human.pedestrian.wheelchair : Wheelchairs. If a person is in the wheelchair, include in the annotation.
movable_object.barrier : Temporary road barrier placed in the scene in order to redirect traffic. Co

# First, putting the images in their respective directories

We need to sort the images by train/val/test and organize them into this YOLO style directory:
```
dataset/
├── train/
│   ├── images/
│   └── labels/
└── val/
    ├── images/
    └── labels/
```


In [20]:
# Let's quickly make those directories
YOLO_TXT_DATASET_ROOT="/home/tensorturtle/DatasetMine/nuImagesYOLOTXT"

# create directories
p = Path(YOLO_TXT_DATASET_ROOT)
p.mkdir(parents=True, exist_ok=True)

train_p = p / 'train'
train_p.mkdir(parents=False, exist_ok=True)

(train_p / 'images').mkdir(parents=False, exist_ok=True)
(train_p / 'labels').mkdir(parents=False, exist_ok=True)

val_p = p / 'val'
val_p.mkdir(parents=False, exist_ok=True)

(val_p / 'images').mkdir(parents=False, exist_ok=True)
(val_p / 'labels').mkdir(parents=False, exist_ok=True)

## Moving Image Files

In [21]:
def move_set(SET_TYPE):
    nuim = NuImages(dataroot=Path(NUIMAGES_ROOT).resolve(), version=f"v1.0-{SET_TYPE}", verbose=False, lazy=True)
    print(f"Moving {SET_TYPE} images")
    for sample in tqdm(nuim.sample):
        move_sample(sample, nuim)

def move_sample(sample, nuim):
    sample_data_token = sample['key_camera_token']
    sample_data = nuim.get("sample_data", sample_data_token)


    origin_jpg_path = Path(NUIMAGES_ROOT) / sample_data['filename']
    assert (origin_jpg_path.exists() & origin_jpg_path.is_file()), f"No image file found at {origin_jpg_path}"

    destination_jpg_dir = Path(YOLO_TXT_DATASET_ROOT) / SET_TYPE / 'images'
    assert (destination_jpg_dir.exists() & destination_jpg_dir.is_dir()), f"No destination directory found at {destination_jpg_dir}"

    if (origin_jpg_path.exists() & destination_jpg_dir.exists()):
        origin_jpg_path.rename(destination_jpg_dir / origin_jpg_path.name )
        


In [22]:
for SET_TYPE in ["val", "train"]:
    move_set(SET_TYPE)

Moving val images


  0%|          | 0/16445 [00:00<?, ?it/s]

100%|██████████| 16445/16445 [00:01<00:00, 14399.81it/s]


Moving train images


100%|██████████| 67279/67279 [00:04<00:00, 14935.36it/s]


Tip: Use `ls | wc -l` to count the number of items in a given directory.

Note: Some images in `samples/CAM...` directories are not moved (they aren't really part of the dataset), and that's ok.

# Convert Annotations

In [350]:
SET_TYPE="train"

In [351]:
nuim = NuImages(dataroot=Path(NUIMAGES_ROOT).resolve(), version=f"v1.0-{SET_TYPE}", verbose=True, lazy=True)

Loading nuImages tables for version v1.0-train...
Done loading in 0.000 seconds (lazy=True).


In [352]:
annotation = nuim.object_ann[10]
annotation

Loaded 557715 object_ann(s) in 2.888s,


{'token': '0000b005d181496a81a62675974bfa1e',
 'category_token': 'fd69059b62a3469fbaef25340c0eab7f',
 'bbox': [1001, 476, 1036, 495],
 'mask': {'size': [900, 1600],
  'counts': 'WFhgazA4a2swMk4yTjFPMTBPMDFPMk4xMDAwMU8wMDAwTzEwMDAwMDAwMDAwMDAxTzAwMDAwMDAwTzJPMDAwMjBOMDAwTzFPMkw0TzFONEtUX10/'},
 'attribute_tokens': ['46dfc76161234ff1a74eff81da7daab0'],
 'sample_data_token': 'fc3bb9f4bfa941a5a9fc3f71939b9279'}

In [353]:
xyXY = annotation['bbox']
height = annotation['mask']['size'][0]
width = annotation['mask']['size'][1]

In [354]:
def PxyXY_to_Nxcycwh(xyXY, width_pixels, height_pixels):
    '''
    Convert from pixel [x_min, y_min, x_max, y_max] to
    normalized [x_center, y_center, width, height]

    Both are left-top origin.
    '''
    # Convert input to a NumPy array
    xyXY = np.asarray(xyXY, dtype=np.float32)
    
    # Calculate the width and height of the bounding box
    box_width_height = xyXY[2:] - xyXY[:2]
    
    # Calculate the center of the bounding box
    box_center = xyXY[:2] + box_width_height / 2.0
    
    # Normalize the center coordinates and dimensions
    normalized = np.hstack((box_center / [width_pixels, height_pixels],
                            box_width_height / [width_pixels, height_pixels]))
    
    return normalized

In [355]:
def test_PxyWX_to_Nxcycwh():
    assert np.all(PxyXY_to_Nxcycwh([0, 0, 200, 100], 200, 100) == np.array([0.5, 0.5, 1, 1]))
    assert np.all(PxyXY_to_Nxcycwh([0, 0, 0, 0], 1920, 1080) == np.array([0., 0., 0., 0.]))
    assert np.all(PxyXY_to_Nxcycwh([1, 1, 3, 2], 5, 4 ) == np.array([0.4, 0.375, 0.4, 0.25]))

In [356]:
test_PxyWX_to_Nxcycwh()

In [357]:
PxyXY_to_Nxcycwh(xyXY, width, height)

array([0.6365625 , 0.53944444, 0.021875  , 0.02111111])

In [358]:
yolo_bbox = list(PxyXY_to_Nxcycwh(xyXY, width, height))

In [359]:
yolo_bbox

[0.6365625, 0.5394444444444444, 0.021875, 0.021111111111111112]

In [360]:
# get category
nuim.get('category', annotation['category_token'])['name']

Loaded 25 category(s) in 0.000s,


'vehicle.car'

In [361]:
if annotation['attribute_tokens']:
    attribute_token = annotation['attribute_tokens'][0]

attribute_token

'46dfc76161234ff1a74eff81da7daab0'

In [362]:
nuim.get('attribute', attribute_token)['name']

Loaded 12 attribute(s) in 0.000s,


'vehicle.stopped'

In [363]:
nuim.list_attributes()


Annotations Name                     Description                                     
     132261 vehicle.parked           Vehicle is stationary (usually for longer durati
      94205 vehicle.moving           Vehicle is moving.                              
      82477 pedestrian.moving        The human is moving.                            
      38239 pedestrian.standing      The human is standing.                          
      22556 cycle.without_rider      There is NO rider on the bicycle or motorcycle. 
      20673 vehicle.stopped          Vehicle, with a driver/rider in/on it, is curren
      13166 pedestrian.sitting_lying The human is sitting or lying down.             
       6660 cycle.with_rider         There is a rider on the bicycle or motorcycle.  
        150 vertical_position.on_gro Object is on the ground plane.                  
        107 vehicle_light.emergency. Vehicle is not flashing emergency lights.       
         32 vehicle_light.emergency. Vehicle is flash

In [364]:
mapping = {
    'animal': None,
    'human.pedestrian.adult': {
        'pedestrian.sitting_lying_down': None,
        'pedestrian.moving': 'pedestrian',
        'pedestrian.standing': 'pedestrian',
    },
    'human.pedestrian.child': {
        'pedestrian.sitting_lying_down': None,
        'pedestrian.moving': 'pedestrian',
        'pedestrian.standing': 'pedestrian',
    },
    'human.pedestrian.construction_worker': 'pedestrian',
    'human.pedestrian.personal_mobility': 'uprightmobility',
    'human.pedestrian.police_officer': 'pedestrian',
    'human.pedestrian.stroller': 'stroller',
    'human.pedestrian.wheelchair': 'wheelchair',
    'movable_object.barrier': None,
    'movable_object.pushable_pullable': None,
    'movable_object.debris': None,
    'movable_object.trafficcone': None,
    'static_object.bicycle_rack': None,
    'vehicle.bicycle': {
        'cycle.with_rider': 'cyclist',
        'cycle.without_rider': 'bicycle',
    },
    'vehicle.bus.bendy': 'bus',
    'vehicle.bus.rigid': 'bus',
    'vehicle.car': 'car',
    'vehicle.construction': None,
    'vehicle.ego': None,
    'vehicle.emergency.ambulance': 'ambulance',
    'vehicle.emergency.police': None,
    'vehicle.motorcycle': {
        'cycle.with_rider': 'motorcyclist',
        'cycle.without_rider': 'motorcycle',
    },
    'vehicle.trailer': None,
    'vehicle.truck': 'truck'
}

def simplify_nuimage_labels(category, attribute):
    assert category in mapping, f"Category: {category} not found in mapping"

    mapp = mapping[category]

    if mapp is None:
        # ignore label
        return None
    elif isinstance(mapp, str):
        # 1-to-1 mapping
        return mapp
    elif attribute in mapp:
        #assert attribute in mapp, f"Attribute: {attribute} not found under category: {mapp}"
        return mapp[attribute]
    else:
        # rare dataset bug where categories that should have an attribute simply doesn't.
        return None

In [365]:
def test_simplify_nuimage_labels():
    assert simplify_nuimage_labels('vehicle.bicycle', 'cycle.without_rider') == 'bicycle'
    assert simplify_nuimage_labels('vehicle.bicycle', 'cycle.with_rider') == 'cyclist'
    assert simplify_nuimage_labels('vehicle.truck', None) == 'truck'

In [366]:
test_simplify_nuimage_labels()

In [367]:
from enum import Enum, auto

class NuImageSimpleCategory(Enum):
    pedestrian = 0
    cyclist = auto()
    car = auto()
    bus = auto()
    truck = auto()
    ambulance = auto()
    uprightmobility = auto()
    stroller = auto()
    wheelchair = auto()
    bicycle = auto()
    motorcyclist = auto()
    motorcycle = auto()

In [368]:
NuImageSimpleCategory['car'].value

2

In [369]:
def convert_set_ann(SET_TYPE):
    nuim = NuImages(dataroot=Path(NUIMAGES_ROOT).resolve(), version=f"v1.0-{SET_TYPE}", verbose=False, lazy=True)

    for annotation in tqdm(nuim.object_ann):
        cat, bbox, filename_no_suffix = convert_annotation(annotation, nuim)
        if cat is None:
            continue
        append_txt(cat, bbox, filename_no_suffix, SET_TYPE)

In [370]:
def convert_annotation(annotation, nuim):
    xyXY = annotation['bbox']

    # odd dataset bug where there is no mask
    if annotation['mask'] is None:
        return None, None, None
    height = annotation['mask']['size'][0]
    width = annotation['mask']['size'][1]
    yolo_bbox = list(PxyXY_to_Nxcycwh(xyXY, width, height))

    nu_cat = nuim.get('category', annotation['category_token'])['name']

    if annotation['attribute_tokens']:
        attribute_token = annotation['attribute_tokens'][0]
        attribute = nuim.get('attribute', attribute_token)['name']
    else:
        attribute = None
    
    yolo_cat = simplify_nuimage_labels(nu_cat, attribute)

    filename_no_suffix = get_filename_no_suffix(annotation, nuim)

    return yolo_cat, yolo_bbox, filename_no_suffix

In [371]:
def get_filename_no_suffix(annotation, nuim):
    '''
    Given a NuImages.ann annotation, return the filename without suffix
    '''
    sample_data_token = annotation['sample_data_token']
    sample_data = nuim.get("sample_data", sample_data_token)
    return Path(sample_data['filename']).with_suffix('').name

In [372]:
def append_txt(cat, bbox, filename_no_suffix, SET_TYPE):
    pa = Path(YOLO_TXT_DATASET_ROOT) / SET_TYPE / 'labels' 
    fi = (pa / filename_no_suffix).with_suffix('.txt')

    cat_index = NuImageSimpleCategory[cat].value

    xc, yc, w, h = bbox

    with open(fi, 'a') as f:
        f.write(f"{cat_index} {xc} {yc} {w} {h}\n")

In [374]:
for set_type in ["train", "val"]:
    print(f"Converting and writing annotations for: {set_type}")
    convert_set_ann(set_type)

100%|██████████| 136074/136074 [00:05<00:00, 25352.47it/s]
0it [00:00, ?it/s]
